# Layer 3 detector: walkthrough

**Objective:** run, step by step, the complete Layer 3 false-data-injection detector and show what each step produces.

```
WegenerNet data → causal features → Extra Trees (one per variable) → calibration → per-variable alert rule → alerts
```

| Step | Data used | Output |
|---|---|---|
| 1. Extract | 24 monthly NetCDF files, 2023–2024 | `data/cache` (162 grid cells × temperature, humidity, rain) |
| 2. Features | nominal Jan–Aug 2023 | `artifacts/feature_extractor.joblib` |
| 3. Train | Jan–Aug 2023 + synthetic attacks | `artifacts/extra_trees.joblib` |
| 4. Calibrate | nominal Sep–Oct 2023 | `artifacts/calibration.npz` |
| 5. Select | Nov–Dec 2023 + synthetic attacks | `artifacts/policy.json` (frozen) |
| 6. Evaluate | 2024 + two attack seeds | `outputs/` (metrics, per-variable, per-family, `RESULTS.md`) |

Every step calls the same functions as `python -m detector.pipeline`. A step whose output already exists is reused
unless `FORCE = True`. CPU only. The reported results are in `results/` (last section).

### Cell objective: set up the session
Finds the project root, makes it the working directory and imports the pipeline. `FORCE = True` reruns steps
whose outputs already exist.

In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "detector").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from detector import pipeline
from detector.utils import OUTPUTS, ARTIFACTS, read_json

FORCE = False

def run(step):
    """Run one pipeline step unless its output already exists (or FORCE is set)."""
    if FORCE or not pipeline.STAGES[step].exists():
        pipeline.RUNNERS[step]()
    else:
        print(f"Reusing {step}: {pipeline.STAGES[step].relative_to(ROOT)}")
print("Project:", ROOT)

## 1. Extract the data
### Cell objective: build (or verify) the compact cache
Reads the WegenerNet NetCDF folders (`WN_L2_HHD_v8_2025_1_UTM_TF1_UTC_2023/` and `…_2024/` at the project root). It
checks units (K → °C), the interval-end time convention and 30-minute continuity, then keeps a 1.5 km sub-grid of
cells with ≥ 95 % clean training data. An existing cache with the same configuration is reused.

In [ ]:
run("extract")
manifest = read_json(ROOT / "data/cache/manifest.json")
{k: manifest[k] for k in ("candidate_sites", "selected_sites", "training_steps")}

### Cell objective: map the selected virtual sites
The spatial features compare each site with its neighbours, so this map is the geometry they rely on.
These are interpolated grid cells, not independent physical sensors.

In [ ]:
sites = pd.read_parquet(ROOT / "data/cache/selection.parquet").query("selected")
ax = sites.plot.scatter("easting_m", "northing_m", s=12, figsize=(7, 4), title=f"{len(sites)} selected sites (EPSG:32633)")
ax.set_aspect("equal"); plt.show()

## 2. Fit the causal feature extractor
### Cell objective: learn the frozen normalisation from nominal Jan–Aug 2023
Fits the persistence/daily forecast blend, the harmonic seasonal reference, robust neighbour offsets and scales,
rate, variability and freeze scales, and the CUSUM normalisation. Every feature uses only current and past values.

In [ ]:
run("features")
from detector.features import FeatureExtractor, FEATURE_NAMES_ALL
extractor = FeatureExtractor.load(pipeline.EXTRACTOR)
print(len(FEATURE_NAMES_ALL), "features:", ", ".join(FEATURE_NAMES_ALL))
print("Selected forecast blend per variable:", extractor.fit_report["selected_blend"])

## 3. Train the Extra Trees models
### Cell objective: sample training rows and fit one classifier per variable
Each training episode is replayed nominal and with 4 seeded attack copies. Per episode and variable it samples up to
1,800 attacked rows and 1,800 nominal rows, a quarter of them *hard* nominal rows with large spatial residuals. The
`pattern`, `mimicry` and `multivariate` families are withheld so selection and testing measure generalisation.

In [ ]:
run("train")
import joblib
models = joblib.load(pipeline.MODELS)
print([m[-1].n_estimators for m in models], "trees per variable")

## 4. Calibrate
### Cell objective: compare scores with nominal Sep–Oct 2023
Stores the sorted nominal scores per variable. A new score becomes the conservative tail value
`p = (1 + #nominal ≥ score) / (n + 1)`: small means unusual. It is not the probability of an attack.

In [ ]:
run("calibrate")
with np.load(pipeline.CALIBRATION) as z:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3))
    for v, ax in enumerate(axes):
        ax.hist(z[str(v)], bins=50, log=True); ax.set_title(["Temperature", "Humidity", "Rain"][v])
plt.tight_layout(); plt.show()

## 5. Select and freeze the alert policy
### Cell objective: choose alpha and confirmation per variable on Nov–Dec 2023
Each validation episode is scored nominal and attacked. For each variable it tries 9 alpha levels × 2 confirmation
rules (`direct`, or `two_of_three`: the current point and one of the two previous points must exceed alpha). The rule
must keep nominal false-alert episodes ≤ 0.05 per stream-day with ≥ 95 % coverage; the best F1 wins. The policy is
frozen together with the hashes of every artifact.

In [ ]:
run("select")
frozen = read_json(pipeline.POLICY)
display(pd.DataFrame(frozen["policy"], index=frozen["variables"]))
display(pd.read_csv(OUTPUTS / "threshold_search.csv").query("feasible").groupby("variable").head(3))

## 6. Evaluate on 2024
### Cell objective: replay 2024 with the frozen detector
Nominal 2024 measures false alarms. Two attack seeds (91000, 191000) measure detection. Nothing is refitted and no
threshold changes. The hashes are verified before scoring.

In [ ]:
run("evaluate")
metrics = read_json(OUTPUTS / "metrics.json")
display(pd.DataFrame({s: metrics[s] for s in ("seed_91000", "seed_191000")}).loc[["precision", "recall_end_to_end", "f1"]])
display(pd.read_csv(OUTPUTS / "families.csv").pivot(index="family", columns="suite", values="point_recall"))

## Use the detector on an episode
### Cell objective: minimal integration example
`Detector()` loads the frozen artifacts (after checking their hashes) and returns, for an episode `[time, site, variable]`,
the calibrated p-values and the boolean alerts. The first 48 steps (one day of warm-up) never alert.

In [ ]:
from detector.model import Detector
from detector.data import load_partition, episodes
_, episode = next(episodes(load_partition("validation")))
p, alerts = Detector()(episode)
print("p shape:", p.shape, "| alerts:", int(alerts.sum()))

## Reported results
### Cell objective: show the results reported in the README
`results/` holds the reported metrics. Running the steps above writes the same tables to `outputs/`, so the two
can be compared directly.

In [ ]:
reported = read_json(ROOT / "results/metrics_2024.json")
display(pd.DataFrame(reported).loc[["precision", "recall_end_to_end", "f1", "false_episodes_per_stream_day"]])
display(pd.read_csv(ROOT / "results/families_2024.csv").query("suite == 'seed_91000'"))